# 03 · Predicción de Demanda
**TFM — Streaming Analytics para Supply Chain | Telecomunicaciones Multi-País**

Autora: Iliana Yazmin Pasos Gallo

---

Este notebook implementa y compara tres modelos de predicción de demanda de unidades por SKU y semana.

| Modelo | Tipo | Rol |
|---|---|---|
| **Regresión Lineal** | Baseline paramétrico | Referencia base interpretable |
| **Random Forest** | Ensemble (bagging) | Robusto a outliers, captura no-linealidades |
| **XGBoost** | Ensemble (boosting) | Mejor precision en series con patrones complejos |

**Variable objetivo:** `DEMANDA` — unidades vendidas por SKU x País x Semana

**Entrada:** `../data/processed/datos_limpios.parquet`

**Salidas:** modelos y metricas en `../outputs/models/`

| Sección | Contenido |
|---|---|
| **0** | Imports y configuración |
| **1** | Carga de datos |
| **2** | Preparación y feature engineering |
| **3** | Split temporal train / test |
| **4** | Entrenamiento de modelos |
| **5** | Evaluación y comparación |
| **6** | Análisis de predicciones |
| **7** | Importancia de features |
| **8** | Análisis por país |
| **9** | Guardar modelos y resultados |
| **10** | Resumen ejecutivo |


## 0 · Imports y Configuración

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import joblib
warnings.filterwarnings('ignore')

from pathlib import Path

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from xgboost import XGBRegressor

# -- Rutas del proyecto -------------------------------------------------------
BASE_DIR   = Path('..')
DATA_PROC  = BASE_DIR / 'data' / 'processed'
OUTPUT_DIR = BASE_DIR / 'outputs' / 'models'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# -- Estilo visual unificado --------------------------------------------------
plt.rcParams.update({
    'figure.facecolor' : '#0f1117',
    'axes.facecolor'   : '#1a1d27',
    'axes.edgecolor'   : '#3a3d4d',
    'axes.labelcolor'  : '#e0e0e0',
    'xtick.color'      : '#a0a0b0',
    'ytick.color'      : '#a0a0b0',
    'text.color'       : '#e0e0e0',
    'grid.color'       : '#2a2d3d',
    'grid.linestyle'   : '--',
    'font.family'      : 'DejaVu Sans',
    'font.size'        : 11,
})

PALETTE = ['#00d4aa', '#6c63ff', '#ff6b6b', '#ffd166']

print('Librerias cargadas correctamente.')


## 1 · Carga de Datos

In [ ]:
df_raw = pd.read_parquet(DATA_PROC / 'datos_limpios.parquet')

print(f'Filas: {df_raw.shape[0]:,} | Columnas: {df_raw.shape[1]}')
print(f'Periodo: {df_raw["FECHA"].min().date()} -- {df_raw["FECHA"].max().date()}')
print(f'Paises : {sorted(df_raw["PAIS"].dropna().unique().tolist())}')
df_raw.head(3)


## 2 · Preparación y Feature Engineering

### 2.1 Filtrar ventas y agregar por semana x SKU x País

In [ ]:
# -- Solo transacciones de venta ----------------------------------------------
df = df_raw[df_raw['TIPO_TRANSACCION'] == 'SALES'].copy()
df['FECHA'] = pd.to_datetime(df['FECHA'])
df['VALUE'] = pd.to_numeric(df['VALUE'], errors='coerce').fillna(0)

print(f'Registros de venta: {len(df):,}')

# -- Agregar por semana x SKU x Pais -----------------------------------------
# PAIS viene del parquet limpio (columna estandarizada por fase1_limpieza.py)
df['YEAR_WEEK'] = df['FECHA'].dt.to_period('W').dt.start_time

df_agg = (
    df.groupby(['YEAR_WEEK', 'CODIGO_SKU', 'FAMILIA', 'MODELO', 'MARCA', 'PAIS'])
      .agg(
          DEMANDA     = ('VALUE',           'sum'),
          NUM_TRANSAC = ('VALUE',           'count'),
          PRECIO_PROM = ('PRECIO_UNITARIO', 'mean'),
          COSTO_PROM  = ('COSTO',           'mean'),
      )
      .reset_index()
      .sort_values(['PAIS', 'CODIGO_SKU', 'YEAR_WEEK'])
)

print(f'Dataset agregado: {len(df_agg):,} filas')
df_agg.head()


### 2.2 Feature Engineering

Los features se construyen en tres grupos:

- **Temporales** — semana del año, mes, trimestre, año y sus versiones ciclicas (seno/coseno)
- **Lags** — demanda de las semanas 1, 2, 4, 8 y 13 semanas atras
- **Ventanas rodantes** — media y desviacion de las ultimas 4 y 8 semanas

Los lags y ventanas se calculan siempre con `shift(1)` para evitar data leakage — el modelo nunca ve la semana actual al predecir.


In [ ]:
def build_features(df):
    d   = df.copy().sort_values(['PAIS', 'CODIGO_SKU', 'YEAR_WEEK'])
    grp = ['PAIS', 'CODIGO_SKU']

    # -- Temporales -----------------------------------------------------------
    d['SEMANA_ANO'] = d['YEAR_WEEK'].dt.isocalendar().week.astype(int)
    d['MES']        = d['YEAR_WEEK'].dt.month
    d['TRIMESTRE']  = d['YEAR_WEEK'].dt.quarter
    d['ANO']        = d['YEAR_WEEK'].dt.year

    # Codificacion ciclica: el modelo entiende que semana 52 esta cerca de la 1
    d['SEMANA_SIN'] = np.sin(2 * np.pi * d['SEMANA_ANO'] / 52)
    d['SEMANA_COS'] = np.cos(2 * np.pi * d['SEMANA_ANO'] / 52)
    d['MES_SIN']    = np.sin(2 * np.pi * d['MES'] / 12)
    d['MES_COS']    = np.cos(2 * np.pi * d['MES'] / 12)

    # -- Lags (semanas previas) -----------------------------------------------
    for lag in [1, 2, 4, 8, 13]:
        d[f'LAG_{lag}W'] = d.groupby(grp)['DEMANDA'].shift(lag)

    # -- Ventanas rodantes ----------------------------------------------------
    d['ROLLING_4W_MEAN'] = (
        d.groupby(grp)['DEMANDA']
         .transform(lambda x: x.shift(1).rolling(4, min_periods=1).mean())
    )
    d['ROLLING_4W_STD'] = (
        d.groupby(grp)['DEMANDA']
         .transform(lambda x: x.shift(1).rolling(4, min_periods=2).std().fillna(0))
    )
    d['ROLLING_8W_MEAN'] = (
        d.groupby(grp)['DEMANDA']
         .transform(lambda x: x.shift(1).rolling(8, min_periods=1).mean())
    )

    # -- Encoding categorico --------------------------------------------------
    for col in ['PAIS', 'FAMILIA', 'MARCA']:
        le = LabelEncoder()
        d[f'{col}_ENC'] = le.fit_transform(d[col].astype(str))

    return d

df_feat = build_features(df_agg)
print(f'Shape tras feature engineering: {df_feat.shape}')
print(f'Features nuevos: {[c for c in df_feat.columns if c not in df_agg.columns]}')


## 3 · Split Temporal Train / Test

Se usa un corte temporal en lugar de un split aleatorio. Entrenar con datos futuros y testear con pasados es un error clasico en series de tiempo que inflaria artificialmente las metricas.

**Criterio:** 80% mas antiguo para entrenamiento, 20% mas reciente para test.


In [ ]:
FEATURE_COLS = [
    'SEMANA_ANO', 'MES', 'TRIMESTRE', 'ANO',
    'SEMANA_SIN', 'SEMANA_COS', 'MES_SIN', 'MES_COS',
    'LAG_1W', 'LAG_2W', 'LAG_4W', 'LAG_8W', 'LAG_13W',
    'ROLLING_4W_MEAN', 'ROLLING_4W_STD', 'ROLLING_8W_MEAN',
    'PRECIO_PROM', 'COSTO_PROM', 'NUM_TRANSAC',
    'PAIS_ENC', 'FAMILIA_ENC', 'MARCA_ENC'
]
TARGET = 'DEMANDA'

# Eliminar filas con NaN en lags (las primeras semanas de cada SKU)
df_model = df_feat[FEATURE_COLS + [TARGET, 'YEAR_WEEK', 'PAIS']].dropna()
print(f'Dataset listo para modelar: {len(df_model):,} filas')

# Corte temporal 80/20
split_date = df_model['YEAR_WEEK'].quantile(0.80, interpolation='nearest')
print(f'Fecha de corte train/test  : {split_date.date()}')

train = df_model[df_model['YEAR_WEEK'] <= split_date]
test  = df_model[df_model['YEAR_WEEK'] >  split_date]

X_train = train[FEATURE_COLS]
y_train = train[TARGET]
X_test  = test[FEATURE_COLS]
y_test  = test[TARGET]

print(f'Train: {len(X_train):,} muestras  ({train["YEAR_WEEK"].min().date()} -- {train["YEAR_WEEK"].max().date()})')
print(f'Test : {len(X_test):,}  muestras  ({test["YEAR_WEEK"].min().date()} -- {test["YEAR_WEEK"].max().date()})')


In [ ]:
# -- Visualizacion del split --------------------------------------------------
fig, ax = plt.subplots(figsize=(13, 4))

weekly = df_model.groupby('YEAR_WEEK')['DEMANDA'].sum()

ax.fill_between(weekly.index[weekly.index <= split_date],
                weekly[weekly.index <= split_date],
                alpha=0.45, color=PALETTE[0], label='Train')
ax.fill_between(weekly.index[weekly.index > split_date],
                weekly[weekly.index > split_date],
                alpha=0.45, color=PALETTE[1], label='Test')
ax.plot(weekly.index, weekly, color='white', lw=1)
ax.axvline(split_date, color=PALETTE[2], lw=2, ls='--',
           label=f'Corte: {split_date.date()}')

ax.set_title('Demanda semanal total -- Split temporal Train / Test', fontsize=13)
ax.set_xlabel('Semana')
ax.set_ylabel('Unidades')
ax.legend()
ax.grid(alpha=0.4)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'split_temporal.png', dpi=150, bbox_inches='tight')
plt.show()


## 4 · Entrenamiento de Modelos

### 4.1 Regresión Lineal — Baseline

Se escala con StandardScaler porque la regresion lineal es sensible a la magnitud de los features. Los otros dos modelos no lo necesitan.


In [ ]:
scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

lr = LinearRegression()
lr.fit(X_train_sc, y_train)

y_pred_lr = np.clip(lr.predict(X_test_sc), 0, None)
print('Regresion Lineal entrenada.')


### 4.2 Random Forest

In [ ]:
rf = RandomForestRegressor(
    n_estimators     = 300,
    max_depth        = 12,
    min_samples_leaf = 5,
    max_features     = 'sqrt',
    n_jobs           = -1,
    random_state     = 42
)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
print('Random Forest entrenado.')


### 4.3 XGBoost

In [ ]:
xgb = XGBRegressor(
    n_estimators     = 500,
    learning_rate    = 0.05,
    max_depth        = 6,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    min_child_weight = 5,
    reg_alpha        = 0.1,
    reg_lambda       = 1.0,
    n_jobs           = -1,
    random_state     = 42,
    verbosity        = 0
)
xgb.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

y_pred_xgb = xgb.predict(X_test)
print('XGBoost entrenado.')


## 5 · Evaluación y Comparación

| Metrica | Qué mide |
|---|---|
| **MAE** | Error promedio en unidades — facil de interpretar para negocio |
| **RMSE** | Penaliza errores grandes mas que el MAE |
| **R²** | Proporcion de varianza explicada por el modelo (1 = perfecto) |
| **MAPE** | Error porcentual — util para comunicar a stakeholders |

Nota: el MAPE puede ser alto en SKUs con demanda de 1-2 unidades donde cualquier error pequeño da un porcentaje elevado. El MAE es la metrica principal para comparar modelos.


In [ ]:
def evaluar(nombre, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mask = y_true > 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    return {'Modelo': nombre, 'MAE': mae, 'RMSE': rmse, 'R2': r2, 'MAPE (%)': mape}

resultados = pd.DataFrame([
    evaluar('Regresion Lineal', y_test, y_pred_lr),
    evaluar('Random Forest',    y_test, y_pred_rf),
    evaluar('XGBoost',          y_test, y_pred_xgb),
]).set_index('Modelo').round(3)

print('-- Metricas en Test Set --')
print(resultados.to_string())


In [ ]:
# -- Grafico comparativo de metricas -----------------------------------------
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
metricas  = ['MAE', 'RMSE', 'R2', 'MAPE (%)']
modelos   = resultados.index.tolist()
etiquetas = ['Lin. Reg.', 'Rnd Forest', 'XGBoost']

for ax, metrica in zip(axes, metricas):
    valores = resultados[metrica].values
    bars    = ax.bar(etiquetas, valores, color=PALETTE[:3],
                     edgecolor='white', linewidth=0.6)

    # Resaltar el mejor
    best = np.argmin(valores) if metrica != 'R2' else np.argmax(valores)
    bars[best].set_edgecolor('#ffd166')
    bars[best].set_linewidth(2.5)

    for bar, val in zip(bars, valores):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + max(valores) * 0.01,
                f'{val:.2f}', ha='center', va='bottom', fontsize=10)

    ax.set_title(metrica, fontsize=13)
    ax.grid(axis='y', alpha=0.4)

fig.suptitle('Comparacion de modelos -- Prediccion de Demanda', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'comparacion_modelos.png', dpi=150, bbox_inches='tight')
plt.show()


## 6 · Análisis de Predicciones

In [ ]:
# -- Scatter real vs predicho para los 3 modelos -----------------------------
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
preds  = [y_pred_lr, y_pred_rf, y_pred_xgb]
labels = ['Regresion Lineal', 'Random Forest', 'XGBoost']

for ax, pred, label, color in zip(axes, preds, labels, PALETTE):
    ax.scatter(y_test, pred, alpha=0.3, s=15, color=color)
    lim = max(y_test.max(), pred.max())
    ax.plot([0, lim], [0, lim], 'w--', lw=1.5, label='Prediccion perfecta')
    ax.set_xlabel('Real')
    ax.set_ylabel('Predicho')
    ax.set_title(label, fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
    r2 = r2_score(y_test, pred)
    ax.text(0.05, 0.92, f'R2 = {r2:.3f}', transform=ax.transAxes,
            color='white', fontsize=11,
            bbox=dict(boxstyle='round', facecolor='#2a2d3d', alpha=0.8))

fig.suptitle('Real vs Predicho por modelo', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'real_vs_predicho.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# -- Serie temporal: demanda real vs mejor modelo ----------------------------
mejor_nombre = resultados['MAE'].idxmin()
pred_map     = {
    'Regresion Lineal': y_pred_lr,
    'Random Forest'   : y_pred_rf,
    'XGBoost'         : y_pred_xgb,
}
y_pred_best = pred_map[mejor_nombre]

test_plot          = test.copy()
test_plot['PRED']  = y_pred_best
weekly_real        = test_plot.groupby('YEAR_WEEK')['DEMANDA'].sum()
weekly_pred        = test_plot.groupby('YEAR_WEEK')['PRED'].sum()

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(weekly_real.index, weekly_real,
        color=PALETTE[0], lw=2.5, marker='o', ms=4, label='Real')
ax.plot(weekly_pred.index, weekly_pred,
        color=PALETTE[1], lw=2, marker='s', ms=4, ls='--',
        label=f'Predicho ({mejor_nombre})')
ax.fill_between(weekly_real.index, weekly_real, weekly_pred,
                alpha=0.15, color=PALETTE[2], label='Error')

ax.set_title(f'Demanda real vs predicha (Test Set) -- {mejor_nombre}', fontsize=13)
ax.set_xlabel('Semana')
ax.set_ylabel('Unidades')
ax.legend()
ax.grid(alpha=0.4)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'serie_real_vs_predicha.png', dpi=150, bbox_inches='tight')
plt.show()


## 7 · Importancia de Features

Los features mas importantes confirman que los **lags y ventanas rodantes** son los mejores predictores — la demanda pasada reciente es la mejor señal para predecir la demanda futura. Esto valida el diseño del feature engineering.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

rf_imp  = pd.Series(rf.feature_importances_,  index=FEATURE_COLS).sort_values()
xgb_imp = pd.Series(xgb.feature_importances_, index=FEATURE_COLS).sort_values()

axes[0].barh(rf_imp.index,  rf_imp.values,  color=PALETTE[0], edgecolor='none')
axes[0].set_title('Random Forest -- Feature Importance', fontsize=13)
axes[0].set_xlabel('Importancia')
axes[0].grid(axis='x', alpha=0.4)

axes[1].barh(xgb_imp.index, xgb_imp.values, color=PALETTE[1], edgecolor='none')
axes[1].set_title('XGBoost -- Feature Importance', fontsize=13)
axes[1].set_xlabel('Importancia')
axes[1].grid(axis='x', alpha=0.4)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 5 features (XGBoost):')
print(xgb_imp.tail(5).sort_values(ascending=False).round(4).to_string())


## 8 · Análisis por País

MAE del mejor modelo desglosado por pais. Cuando se incorporen los datos de todos los paises esta seccion mostrara si el modelo generaliza bien entre mercados o si necesita entrenamiento por separado.


In [ ]:
test_eval              = test[['YEAR_WEEK', 'PAIS', TARGET]].copy()
test_eval['PRED']      = y_pred_best
test_eval['ABS_ERROR'] = np.abs(test_eval[TARGET] - test_eval['PRED'])

pais_metrics = (
    test_eval.groupby('PAIS')
    .apply(lambda g: pd.Series({
        'MAE'       : mean_absolute_error(g[TARGET], g['PRED']),
        'RMSE'      : np.sqrt(mean_squared_error(g[TARGET], g['PRED'])),
        'R2'        : r2_score(g[TARGET], g['PRED']),
        'N_muestras': len(g)
    }))
    .reset_index()
    .sort_values('MAE')
)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(pais_metrics['PAIS'], pais_metrics['MAE'],
              color=PALETTE[0], edgecolor='white', linewidth=0.6)
for bar, val in zip(bars, pais_metrics['MAE']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
            f'{val:.1f}', ha='center', va='bottom', fontsize=10)

ax.set_title(f'MAE por pais -- {mejor_nombre}', fontsize=13)
ax.set_ylabel('MAE (unidades)')
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'mae_por_pais.png', dpi=150, bbox_inches='tight')
plt.show()

print(pais_metrics.round(3).to_string(index=False))


## 9 · Guardar Modelos y Resultados

In [ ]:
# -- Modelos entrenados -------------------------------------------------------
joblib.dump(lr,     OUTPUT_DIR / 'model_linear_regression.pkl')
joblib.dump(rf,     OUTPUT_DIR / 'model_random_forest.pkl')
joblib.dump(xgb,    OUTPUT_DIR / 'model_xgboost.pkl')
joblib.dump(scaler, OUTPUT_DIR / 'scaler_linear.pkl')

# -- Metricas -----------------------------------------------------------------
resultados.to_csv(OUTPUT_DIR / 'metricas_modelos.csv')
pais_metrics.to_csv(OUTPUT_DIR / 'metricas_por_pais.csv', index=False)

# -- Predicciones del mejor modelo --------------------------------------------
test_eval.to_parquet(OUTPUT_DIR / 'predicciones_test.parquet', index=False)

print('Modelos y resultados guardados en:', OUTPUT_DIR)
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f'  {f.name}')


## 10 · Resumen Ejecutivo

In [ ]:
mejor_r2  = resultados['R2'].max()
mejor_mae = resultados['MAE'].min()

print('=' * 55)
print('  RESUMEN -- PREDICCION DE DEMANDA')
print('=' * 55)
print(f'Dataset      : {len(df_model):,} observaciones')
print(f'Train        : {train["YEAR_WEEK"].min().date()} -- {train["YEAR_WEEK"].max().date()}  ({len(X_train):,} muestras)')
print(f'Test         : {test["YEAR_WEEK"].min().date()} -- {test["YEAR_WEEK"].max().date()}  ({len(X_test):,} muestras)')
print(f'Features     : {len(FEATURE_COLS)}')
print('-' * 55)
print(resultados[['MAE', 'RMSE', 'R2', 'MAPE (%)']].to_string())
print('-' * 55)
print(f'Mejor modelo : {mejor_nombre}')
print(f'  MAE        : {mejor_mae:.2f} unidades')
print(f'  R2         : {mejor_r2:.3f}')
print('=' * 55)


## 11 · Predicción por Gama — Modelos Nuevos sin Historial

Cuando un SKU nuevo llega al catálogo (ej. Samsung A56 reemplaza al A54),
no tiene historial propio para construir los lags. Este modelo predice la
demanda a nivel de **MARCA x TIER x PAIS** usando el historial agregado
de toda la gama, de modo que cualquier modelo nuevo puede heredar esa
prediccion como punto de partida.

**Ejemplo de uso:**
- Samsung A54 → EOL, sin ventas futuras
- Samsung A56 → nuevo, sin historial
- El modelo predice: *Samsung MID Paraguay vende ~X uds/semana*
- El A56 hereda esa prediccion hasta acumular historial propio

**Diferencia con el modelo principal:**

| | Modelo SKU (secciones 4-9) | Modelo Gama (esta seccion) |
|---|---|---|
| Granularidad | SKU x Semana | MARCA x TIER x Semana |
| Requiere historial | Si, min 4 semanas | No |
| Uso | SKUs activos conocidos | SKUs nuevos o sin historial |


### 11.1 Agregar por MARCA x TIER x PAIS x Semana

In [ ]:
# -- Agregar ventas por gama (MARCA x TIER x PAIS) --------------------------
df_gama = (
    df[['YEAR_WEEK', 'MARCA', 'TIER', 'PAIS', 'VALUE', 'PRECIO_UNITARIO', 'COSTO']]
    .copy()
    .dropna(subset=['MARCA', 'TIER'])
)

df_gama_agg = (
    df_gama
    .groupby(['YEAR_WEEK', 'MARCA', 'TIER', 'PAIS'])
    .agg(
        DEMANDA_GAMA  = ('VALUE',           'sum'),
        NUM_SKUS      = ('VALUE',           'count'),
        PRECIO_PROM   = ('PRECIO_UNITARIO', 'mean'),
        COSTO_PROM    = ('COSTO',           'mean'),
    )
    .reset_index()
    .sort_values(['PAIS', 'MARCA', 'TIER', 'YEAR_WEEK'])
)

print(f'Dataset de gama: {len(df_gama_agg):,} filas')
print(f'Combinaciones MARCA x TIER x PAIS: {df_gama_agg.groupby(["MARCA","TIER","PAIS"]).ngroups}')
df_gama_agg.head()


### 11.2 Feature Engineering por Gama

In [ ]:
def build_features_gama(df: pd.DataFrame) -> pd.DataFrame:
    d   = df.copy().sort_values(['PAIS', 'MARCA', 'TIER', 'YEAR_WEEK'])
    grp = ['PAIS', 'MARCA', 'TIER']

    # -- Temporales -----------------------------------------------------------
    d['SEMANA_ANO'] = d['YEAR_WEEK'].dt.isocalendar().week.astype(int)
    d['MES']        = d['YEAR_WEEK'].dt.month
    d['TRIMESTRE']  = d['YEAR_WEEK'].dt.quarter
    d['ANO']        = d['YEAR_WEEK'].dt.year

    d['SEMANA_SIN'] = np.sin(2 * np.pi * d['SEMANA_ANO'] / 52)
    d['SEMANA_COS'] = np.cos(2 * np.pi * d['SEMANA_ANO'] / 52)
    d['MES_SIN']    = np.sin(2 * np.pi * d['MES'] / 12)
    d['MES_COS']    = np.cos(2 * np.pi * d['MES'] / 12)

    # -- Lags de la gama ------------------------------------------------------
    for lag in [1, 2, 4, 8, 13]:
        d[f'LAG_{lag}W'] = d.groupby(grp)['DEMANDA_GAMA'].shift(lag)

    # -- Ventanas rodantes ----------------------------------------------------
    d['ROLLING_4W_MEAN'] = (
        d.groupby(grp)['DEMANDA_GAMA']
         .transform(lambda x: x.shift(1).rolling(4, min_periods=1).mean())
    )
    d['ROLLING_4W_STD'] = (
        d.groupby(grp)['DEMANDA_GAMA']
         .transform(lambda x: x.shift(1).rolling(4, min_periods=2).std().fillna(0))
    )
    d['ROLLING_8W_MEAN'] = (
        d.groupby(grp)['DEMANDA_GAMA']
         .transform(lambda x: x.shift(1).rolling(8, min_periods=1).mean())
    )

    # -- Encoding categorico --------------------------------------------------
    for col in ['PAIS', 'MARCA', 'TIER']:
        le = LabelEncoder()
        d[f'{col}_ENC'] = le.fit_transform(d[col].astype(str))

    return d

df_gama_feat = build_features_gama(df_gama_agg)
print(f'Shape tras feature engineering: {df_gama_feat.shape}')


### 11.3 Split y Entrenamiento

In [ ]:
FEATURE_COLS_GAMA = [
    'SEMANA_ANO', 'MES', 'TRIMESTRE', 'ANO',
    'SEMANA_SIN', 'SEMANA_COS', 'MES_SIN', 'MES_COS',
    'LAG_1W', 'LAG_2W', 'LAG_4W', 'LAG_8W', 'LAG_13W',
    'ROLLING_4W_MEAN', 'ROLLING_4W_STD', 'ROLLING_8W_MEAN',
    'PRECIO_PROM', 'COSTO_PROM', 'NUM_SKUS',
    'PAIS_ENC', 'MARCA_ENC', 'TIER_ENC',
]
TARGET_GAMA = 'DEMANDA_GAMA'

df_gama_model = df_gama_feat[FEATURE_COLS_GAMA + [TARGET_GAMA, 'YEAR_WEEK', 'PAIS', 'MARCA', 'TIER']].dropna()
print(f'Dataset listo: {len(df_gama_model):,} filas')

# Split temporal 80/20
split_gama = df_gama_model['YEAR_WEEK'].quantile(0.80, interpolation='nearest')
print(f'Corte train/test: {split_gama.date()}')

train_g = df_gama_model[df_gama_model['YEAR_WEEK'] <= split_gama]
test_g  = df_gama_model[df_gama_model['YEAR_WEEK'] >  split_gama]

X_train_g = train_g[FEATURE_COLS_GAMA]
y_train_g = train_g[TARGET_GAMA]
X_test_g  = test_g[FEATURE_COLS_GAMA]
y_test_g  = test_g[TARGET_GAMA]

print(f'Train: {len(X_train_g):,} | Test: {len(X_test_g):,}')

# XGBoost para la gama (mismos hiperparametros)
xgb_gama = XGBRegressor(
    n_estimators     = 500,
    learning_rate    = 0.05,
    max_depth        = 6,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    min_child_weight = 3,
    reg_alpha        = 0.1,
    reg_lambda       = 1.0,
    n_jobs           = -1,
    random_state     = 42,
    verbosity        = 0,
)
xgb_gama.fit(X_train_g, y_train_g, eval_set=[(X_test_g, y_test_g)], verbose=False)
y_pred_gama = xgb_gama.predict(X_test_g)

print('Modelo de gama entrenado.')


### 11.4 Evaluación

In [ ]:
res_gama = pd.DataFrame([
    evaluar('XGBoost Gama', y_test_g, y_pred_gama)
]).set_index('Modelo').round(3)

print('-- Metricas modelo de gama --')
print(res_gama.to_string())
print()

# Comparar con el modelo SKU
print('-- Comparativa --')
print(f'  Modelo SKU  MAE: {resultados.loc["XGBoost", "MAE"]:.2f} uds/SKU/semana')
print(f'  Modelo Gama MAE: {res_gama.loc["XGBoost Gama", "MAE"]:.2f} uds/gama/semana')
print()
print('Nota: el MAE de gama es mayor en valor absoluto porque predice')
print('la demanda total de toda la gama, no de un SKU individual.')


### 11.5 Tabla de Referencia por Gama

In [ ]:
# -- Tabla de demanda promedio por gama para uso en produccion ---------------
# Esta tabla se usa cuando llega un SKU nuevo sin historial:
# se busca su MARCA x TIER x PAIS y se toma la demanda de referencia

tabla_referencia_gama = (
    df_gama_agg
    .groupby(['PAIS', 'MARCA', 'TIER'])
    .agg(
        DEMANDA_SEMANAL_PROM = ('DEMANDA_GAMA', 'mean'),
        DEMANDA_SEMANAL_MAX  = ('DEMANDA_GAMA', 'max'),
        DEMANDA_SEMANAL_STD  = ('DEMANDA_GAMA', 'std'),
        SEMANAS_CON_DATOS    = ('DEMANDA_GAMA', 'count'),
        PRECIO_PROM          = ('PRECIO_PROM',  'mean'),
    )
    .round(1)
    .reset_index()
    .sort_values(['PAIS', 'MARCA', 'DEMANDA_SEMANAL_PROM'], ascending=[True, True, False])
)

print('Tabla de referencia por gama:')
print(tabla_referencia_gama.to_string(index=False))


### 11.6 Guardar Modelo de Gama

In [ ]:
# -- Guardar modelo y tabla de referencia ------------------------------------
joblib.dump(xgb_gama, OUTPUT_DIR / 'model_xgboost_gama.pkl')

tabla_referencia_gama.to_csv(OUTPUT_DIR / 'tabla_referencia_gama.csv', index=False)
tabla_referencia_gama.to_parquet(OUTPUT_DIR / 'tabla_referencia_gama.parquet', index=False)

# Agregar al Excel de resultados
with pd.ExcelWriter(OUTPUT_DIR / 'reporte_modelos.xlsx',
                    engine='openpyxl', mode='a',
                    if_sheet_exists='replace') as writer:
    res_gama.to_excel(writer, sheet_name='metricas_gama')
    tabla_referencia_gama.to_excel(writer, sheet_name='referencia_gama', index=False)

print('Modelo de gama guardado:')
print(f'  model_xgboost_gama.pkl')
print(f'  tabla_referencia_gama.csv  ({len(tabla_referencia_gama)} combinaciones MARCA x TIER x PAIS)')

# -- Como usar en produccion para un SKU nuevo --------------------------------
print()
print('-- Ejemplo de uso para SKU nuevo --')
print('Nuevo modelo: Samsung MID Paraguay')

ejemplo = tabla_referencia_gama[
    (tabla_referencia_gama['PAIS']  == 'PY') &
    (tabla_referencia_gama['MARCA'] == 'SAMSUNG') &
    (tabla_referencia_gama['TIER']  == 'MID')
]
if not ejemplo.empty:
    dem = ejemplo['DEMANDA_SEMANAL_PROM'].values[0]
    std = ejemplo['DEMANDA_SEMANAL_STD'].values[0]
    print(f'  Demanda esperada : {dem:.1f} uds/semana')
    print(f'  Rango probable   : {max(0, dem-std):.1f} -- {dem+std:.1f} uds/semana')
    print(f'  Stock sugerido   : {dem * 4:.0f} uds (4 semanas de cobertura)')
